**Лабораторная работа №14**

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.util import img_as_float
from sklearn.cluster import KMeans

%matplotlib inline

1. Загрузите картинку parrots.jpg. Преобразуйте изображение, приведя все значения в интервал от 0 до 1. Для этого можно воспользоваться функцией img_as_float из модуля skimage. Обратите внимание на этот шаг, так как при работе с исходным изображением вы получите некорректный результат.


In [2]:
try:
    image = imread('parrots.jpg')
except FileNotFoundError:
    from skimage.data import astronaut
    image = astronaut()
    print("Файл parrots.jpg не найден, используется изображение 'astronaut'")

image_float = img_as_float(image)
print("Размер изображения:", image_float.shape)
print("Диапазон значений:", image_float.min(), "-", image_float.max())

Размер изображения: (474, 713, 3)
Диапазон значений: 0.0 - 1.0


2. Создайте матрицу объекты-признаки: характеризуйте каждый пиксель тремя координатами - значениями интенсивности в пространстве RGB.

In [3]:
h, w, c = image_float.shape

pixels = image_float.reshape(-1, 3)
print(f"Количество пикселей: {pixels.shape[0]}")

Количество пикселей: 337962


3. Запустите алгоритм K-Means с параметрами init=’k-means++’ и random_state=241. После выделения кластеров все пиксели, отнесенные в один кластер, попробуйте заполнить двумя способами: медианным и средним цветом по кластеру.

4. Измерьте качество получившейся сегментации с помощью метрики PSNR.

5. Найдите минимальное количество кластеров, при котором значение PSNR выше 20. Это число и будет ответом в данной задаче.

In [4]:
def psnr(original, compressed):
    mse = np.mean((original - compressed) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * math.log10(1.0 / mse)

In [6]:
max_k = 20
best_k_mean = None
best_k_median = None
psnr_threshold = 20

for k in range(1, max_k + 1):
    print(f"Обработка k = {k}")
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=241, n_init=10)
    kmeans.fit(pixels)
    labels = kmeans.labels_
    centers = kmeans.cluster_centers_

    recovered_mean = centers[labels].reshape(h, w, c)
    psnr_mean = psnr(image_float, recovered_mean)

    median_colors = np.zeros((k, 3))
    for cluster_id in range(k):
        cluster_pixels = pixels[labels == cluster_id]
        if len(cluster_pixels) > 0:
            median_colors[cluster_id] = np.median(cluster_pixels, axis=0)
        else:
            median_colors[cluster_id] = centers[cluster_id]
    recovered_median = median_colors[labels].reshape(h, w, c)
    psnr_median = psnr(image_float, recovered_median)

    print(f"  PSNR (среднее): {psnr_mean:.2f}, PSNR (медиана): {psnr_median:.2f}")

    if psnr_mean > psnr_threshold or psnr_median > psnr_threshold:
        best_k = k
        print(f"  -> Достигнут порог {psnr_threshold} при k = {k}")
        break
else:
    best_k = None
    print(f"Порог {psnr_threshold} не достигнут ни при одном k до {max_k}")

Обработка k = 1
  PSNR (среднее): 9.84, PSNR (медиана): 9.46
Обработка k = 2
  PSNR (среднее): 12.11, PSNR (медиана): 11.68
Обработка k = 3
  PSNR (среднее): 13.18, PSNR (медиана): 12.80
Обработка k = 4
  PSNR (среднее): 14.39, PSNR (медиана): 14.04
Обработка k = 5
  PSNR (среднее): 15.56, PSNR (медиана): 15.21
Обработка k = 6
  PSNR (среднее): 16.57, PSNR (медиана): 16.08
Обработка k = 7
  PSNR (среднее): 17.67, PSNR (медиана): 17.37
Обработка k = 8
  PSNR (среднее): 18.47, PSNR (медиана): 18.18
Обработка k = 9
  PSNR (среднее): 19.14, PSNR (медиана): 18.85
Обработка k = 10
  PSNR (среднее): 19.67, PSNR (медиана): 19.39
Обработка k = 11
  PSNR (среднее): 20.16, PSNR (медиана): 19.89
  -> Достигнут порог 20 при k = 11


In [7]:
if best_k is not None:
    print(f"\nМинимальное количество кластеров с PSNR > {psnr_threshold}: {best_k}")


Минимальное количество кластеров с PSNR > 20: 11
